## 01 — What We Built

Before moving on to the library version, we take stock.

Good engineers do not just ship and move on — they understand what they built, where it breaks, and what it would cost to fix those things. This notebook measures the handbuilt system honestly.

## What We Built — Component Inventory

| Component | Where | What it does |
|-----------|-------|--------------|
| **Douglas-Peucker** | Module 01 | Reduces point count per line segment |
| **LOD pipeline** | Module 02 | Produces 4 simplified GeoJSON files |
| **Bbox computation** | Module 03 | Gets the extent of any feature |
| **Bbox intersection** | Module 03 | Tests if a feature overlaps the viewport |
| **Uniform grid index** | Module 04 | Buckets features for fast viewport queries |
| **LOD decision function** | Module 05 | Selects the right file by zoom level |
| **Live map viewer** | Module 06 | Wires everything into an interactive display |

Each component was built from scratch. We understand every line.

## Measuring the System

In [1]:
import json
import time
from pathlib import Path

def find_data_dir():
    """Find the Data Manager data directory from the repo root or this notebook folder."""
    for base in [Path.cwd(), *Path.cwd().parents]:
        for candidate in [
            base / "data",
            base / "Assignments_Completed" / "03-Data_Manager" / "data",
        ]:
            if (candidate / "lod").exists() and (candidate / "ne_10m_railroads.geojson").exists():
                return candidate
    raise FileNotFoundError("Could not find Assignments_Completed/03-Data_Manager/data")

data_dir = find_data_dir()
lod_dir  = data_dir / "lod"
raw_path = data_dir / "ne_10m_railroads.geojson"

lod_files = {
    "coarse":     "railroads_coarse.geojson",
    "medium":     "railroads_medium.geojson",
    "fine":       "railroads_fine.geojson",
    "extra_fine": "railroads_extra_fine.geojson",
}

print(f"{'File':<28} {'Size (MB)':>10} {'Features':>10} {'Total pts':>12} {'Load (s)':>10}")
print("-" * 75)

for label, filename in [("original", None)] + list(lod_files.items()):
    path = raw_path if filename is None else lod_dir / filename
    t0 = time.perf_counter()
    with open(path) as f:
        data = json.load(f)
    load_time = time.perf_counter() - t0
    feats = data["features"]
    n_pts = sum(len(f["geometry"]["coordinates"]) for f in feats)
    size  = path.stat().st_size / 1_000_000
    print(f"{label:<28} {size:>10.2f} {len(feats):>10,} {n_pts:>12,} {load_time:>10.3f}")

File                          Size (MB)   Features    Total pts   Load (s)
---------------------------------------------------------------------------
original                          39.60     25,413    1,396,480      0.404
coarse                             1.08      2,845        5,690      0.008
medium                             9.66     25,413       55,513      0.085
fine                              11.34     25,413      124,879      0.089
extra_fine                        18.98     25,413      441,871      0.168


## Where the System Still Hurts

The viewer works. But it has real limitations. Let's name them honestly.

### Pain Point 1 — Startup Cost

Every session, we load 4 files and build 4 grid indexes. This takes several seconds before the map is usable.

A tile server has no startup cost — tiles are pre-built and stored. The server just reads a file from a database and sends it.

In [2]:
def feature_bbox(feature):
    coords = feature["geometry"]["coordinates"]
    lons = [c[0] for c in coords]
    lats = [c[1] for c in coords]
    return [min(lons), min(lats), max(lons), max(lats)]

class GridIndex:
    CELL_SIZE = 10.0
    def __init__(self): self.cells = {}
    def _cells(self, bbox):
        lo, la, hi, ha = bbox; cs = self.CELL_SIZE
        return [(c, r) for c in range(int((lo+180)/cs), int((hi+180)/cs)+1)
                       for r in range(int((la+ 90)/cs), int((ha+ 90)/cs)+1)]
    def build(self, features):
        self.cells = {}
        for i, f in enumerate(features):
            for cell in self._cells(feature_bbox(f)): self.cells.setdefault(cell,[]).append((i,f))

total_startup = 0
for filename in lod_files.values():
    t0 = time.perf_counter()
    with open(lod_dir / filename) as f:
        feats = json.load(f)["features"]
    idx = GridIndex()
    idx.build(feats)
    elapsed = time.perf_counter() - t0
    total_startup += elapsed

print(f"Total startup time (load + index build): {total_startup:.2f}s")

Total startup time (load + index build): 0.49s


### Pain Point 2 — GeoJSON Is Verbose

GeoJSON is human-readable text. Every coordinate is stored as a decimal number string. A production mapping pipeline uses **binary encoding** (Mapbox Vector Tiles, MVT) which stores coordinates as integers relative to the tile origin — 5–10× smaller than equivalent GeoJSON and much faster to parse.

In [3]:
# Rough estimate: how large would our files be in a binary format?
# MVT stores coordinates as 2-byte integers per axis
# GeoJSON stores them as ~8-character float strings

GEOJSON_BYTES_PER_COORD = 16   # avg chars for [lon, lat] pair including punctuation
MVT_BYTES_PER_COORD     = 4    # 2 bytes each for x, y as zigzag-encoded varint

for filename in lod_files.values():
    with open(lod_dir / filename) as f:
        feats = json.load(f)["features"]
    total_pts = sum(len(f["geometry"]["coordinates"]) for f in feats)
    actual_mb  = (lod_dir / filename).stat().st_size / 1_000_000
    est_mvt_mb = total_pts * MVT_BYTES_PER_COORD / 1_000_000
    print(f"{filename:<38} actual: {actual_mb:.2f}MB   est MVT: {est_mvt_mb:.2f}MB")

railroads_coarse.geojson               actual: 1.08MB   est MVT: 0.02MB
railroads_medium.geojson               actual: 9.66MB   est MVT: 0.22MB
railroads_fine.geojson                 actual: 11.34MB   est MVT: 0.50MB
railroads_extra_fine.geojson           actual: 18.98MB   est MVT: 1.77MB


### Pain Point 3 — The Whole File Is Always Resident

To query the fine LOD for Paris, we load the entire `railroads_fine.geojson` into memory — including Australia, South America, and Russia. A tile system would read only the Paris tile from a database, never touching the rest.

Our grid index helps at query time, but the full file still had to load first.

### Pain Point 4 — No Partial Load or Streaming

When the user pans to a new region, we re-query the index immediately — but the data was all loaded at startup. A tile server streams only the tiles the user actually views. If the user never visits Australia, those tiles are never fetched.

## The Decision Inventory

Every system embeds design decisions. Here are ours, stated explicitly:

| Decision | What we chose | What we gave up |
|----------|--------------|------------------|
| File format | GeoJSON (text) | Binary efficiency |
| Simplification algorithm | Douglas-Peucker via Shapely | Topology-preserving alternatives |
| LOD levels | 4 fixed levels | Continuous zoom-adaptive detail |
| Coarse filter | scalerank ≤ 4 | Coverage in scalerank 5+ regions |
| Spatial index | Uniform 10° grid | Adaptive indexes (R-tree, quadtree) |
| Culling granularity | Feature bbox | True geometry intersection |
| Transition policy | Fixed zoom thresholds | Hysteresis (implemented but not used in final viewer) |
| Memory model | All data loaded at startup | Lazy / tile-based loading |

None of these decisions are wrong. They are appropriate for a teaching system built from scratch. A production system makes different choices for different reasons.

## Exercise A

Measure the peak memory usage of the viewer at startup (after all 4 LOD files are loaded and all 4 indexes are built).

Use Python's `tracemalloc` module:

```python
import tracemalloc
tracemalloc.start()
# ... load and build ...
current, peak = tracemalloc.get_traced_memory()
print(f"Peak memory: {peak / 1_000_000:.1f} MB")
```

How does this compare to just reading the four files without building indexes?

In [4]:
# Measure peak memory: (a) loading files only, (b) loading + building indexes
import tracemalloc

def measure_load_only():
    tracemalloc.start()
    loaded = {}
    for label, filename in lod_files.items():
        with open(lod_dir / filename) as f:
            loaded[label] = json.load(f)["features"]
    current, peak = tracemalloc.get_traced_memory()
    tracemalloc.stop()
    return current, peak, loaded


def measure_load_and_index():
    tracemalloc.start()
    indexes = {}
    for label, filename in lod_files.items():
        with open(lod_dir / filename) as f:
            features = json.load(f)["features"]
        idx = GridIndex()
        idx.build(features)
        indexes[label] = idx
    current, peak = tracemalloc.get_traced_memory()
    tracemalloc.stop()
    return current, peak, indexes


load_current, load_peak, loaded_features = measure_load_only()
index_current, index_peak, built_indexes = measure_load_and_index()

extra_peak = index_peak - load_peak
ratio = index_peak / load_peak if load_peak else float("inf")

print(f"Load only peak:        {load_peak / 1_000_000:.1f} MB")
print(f"Load + indexes peak:   {index_peak / 1_000_000:.1f} MB")
print(f"Index overhead peak:   {extra_peak / 1_000_000:.1f} MB")
print(f"Load + indexes uses about {ratio:.2f}x the peak memory of loading only.")

Load only peak:        208.4 MB
Load + indexes peak:   214.0 MB
Index overhead peak:   5.6 MB
Load + indexes uses about 1.03x the peak memory of loading only.


## Exercise B

Write a short summary (8–12 sentences) of the Railroad LOD system as if you were presenting it to a team that had never seen it.

Cover:
- What problem it solves
- What the four major components are
- What the main performance tradeoffs are
- What you would change if this needed to serve 10 million users instead of one notebook

Write it in the cell below as markdown.

The Railroad LOD system is a notebook-built map pipeline for displaying a large railroad GeoJSON dataset without drawing every original coordinate at every zoom level. It solves the problem of keeping an interactive map responsive by reducing geometry detail, loading an appropriate level of detail, and querying only features near the current viewport. The first major component is Douglas-Peucker simplification, which removes points while preserving the overall shape of railroad lines. The second component is the LOD pipeline, which writes four separate GeoJSON files: coarse, medium, fine, and extra fine. The third component is spatial indexing, where each feature is assigned to a uniform 10-degree grid so viewport queries do not scan every feature. The fourth component is the live viewer, which chooses an LOD from the current zoom level and displays only the indexed features near the map bounds. The main performance win is faster rendering and querying inside the notebook. The main tradeoff is that startup still loads every LOD file and builds every index before the viewer is ready. GeoJSON also remains bulky because it stores coordinates as text, so the files are larger and slower to parse than a binary tile format. If this system needed to serve 10 million users, I would replace the all-in-memory notebook design with prebuilt vector tiles, a tile server or static tile hosting, binary MVT encoding, caching, and lazy loading so each client downloads only the tiles it actually views.

## Check Your Understanding

We identified four pain points: startup cost, verbose format, whole-file loading, and no streaming.

Rank them from **most impactful** to **least impactful** for a user on a slow connection (e.g., mobile data). Justify your ranking in 2–3 sentences.

**Answer:**

Most impactful to least impactful: **no streaming**, **whole-file loading**, **verbose format**, then **startup cost**. On a slow connection, the biggest problem is downloading data the user may never view, so no streaming and whole-file loading dominate the experience. GeoJSON's verbosity makes that download larger, while startup cost is partly a result of those earlier choices and matters after the data has already arrived.

## Next

In [Module 07 — The Library Version](../07-The_Library_Version/README.md), we hand the problem to `tippecanoe` and see how a professional tool addresses every one of these pain points.